# Notebook 23 — Class-1 entry precision and abstention

**Goal:** Evaluate the q=7 target (`[2,2,0,0,1,1,1]`) as a selective **long-entry** signal. Compare balanced and unweighted logistic/forest models; select probability thresholds using **inner folds only**. The three outer development folds estimate unseen performance.

**Caution:** The old final holdout and prospective ledger are not touched. Unweighted variants are diagnostics; balanced classifiers remain available. Entry returns here are simple one-bar estimates, **not** executable P&L.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=="notebooks": ROOT=ROOT.parent
if not (ROOT/"src").exists(): raise RuntimeError("Run from repository root or notebooks directory")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.entry_signal import run_entry_signal,THRESHOLDS,CONFIGS
print("Repository:",ROOT)

In [ ]:
N_OUTER=3
N_INNER=2
MIN_COVERAGE=.05
ASSUMED_ROUND_TRIP_COST_BPS=10
MAX_TRAIN_ROWS=25000
TREE_TRAIN_ROWS=15000
OUTPUT=ROOT/"artifacts"/"entry_precision"/"1d"
required=("master_labeled.parquet","latest_inference.parquet","transformers.joblib","manifest.json")
locations=[ROOT/"artifacts"/"master_tables"/"1d",ROOT/"notebooks"/"artifacts"/"master_tables"/"1d"]
found=next((d for d in locations if (d/"master_labeled.parquet").exists()),None)
if found is None: raise FileNotFoundError("Run Notebook 03 to create master_labeled.parquet")
master,latest,bundle,manifest=load_master(found)
print("Master rows:",len(master),"Candidates:",CONFIGS,"Thresholds:",THRESHOLDS)

## Chronological threshold selection

Within each outer training period, the two inner folds choose the model and threshold maximizing **class-1 precision**, subject to at least 5% prediction coverage. Outer periods are evaluated once and are never used to tune thresholds. A class-1 prediction must also be the model's highest-probability class.

In [ ]:
inner,outer,selections,cutoff=run_entry_signal(master,n_outer=N_OUTER,n_inner=N_INNER,min_coverage=MIN_COVERAGE,max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,cost_bps=ASSUMED_ROUND_TRIP_COST_BPS,output_dir=OUTPUT)
print("Development cutoff:",cutoff)
display(selections)
display(outer)

## Decision diagnostics

Compare entry precision with the actual Up-class base rate and compare coverage with the minimum. **High precision at very low coverage is not sufficient**. The cost-adjusted return estimate assumes entering at the bar close and exiting one bar later; it ignores market impact and order fill uncertainty.

In [ ]:
fields=["outer_fold","config","threshold","n","entries","coverage","precision_up","base_up_rate","precision_lift","recall_up","overall_accuracy","balanced_accuracy","mean_entry_gross_return","mean_entry_net_return","positive_net_return_rate"]
display(outer[[c for c in fields if c in outer.columns]])
if "precision_up" in outer: print("Mean outer class-1 precision:",round(outer.precision_up.mean(),4))
if "coverage" in outer: print("Mean outer entry coverage:",round(outer.coverage.mean(),4))
print("Saved:",OUTPUT)

## Interpretation and next steps

- Do **not** promote any model on the basis of the same three outer folds used to design later experiments.
- Q5–Q7 are **relative** return bins, not guaranteed positive net returns. Compare the positive net-return rate and mean return with realistic costs.
- The unweighted training variants are a diagnostic comparison; preserve `class_weight="balanced"` for the user-requested balanced baseline.
- The frozen q=3 prospective model and ledger are intentionally unchanged. A future q=7 shadow model needs a new version and genuinely future predictions.